# ⭐ Lesson 6: Catch the Star!

Py has scattered a star somewhere on the screen and wants **you** to
steer them to catch it! You already know everything you need for this:
`forward()`, `left()`, `right()`, and checking distances with `if`.

This is a real, playable game — built entirely out of the Python you've
already learned.

In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join("..", "..")))
from helpers.kid_tools import celebrate, try_again, check_answer, quiz, badge


## How to play

Run the next cell to start the game, then use the buttons underneath it:

* **⬆️ Forward** — move Py forward
* **↩️ Turn Left** / **↪️ Turn Right** — turn Py in place
* **🔄 New Game** — reset your score and try again

The scoreboard tells you how far away the star is. Get close enough and
you **catch** it — a new star appears and your score goes up!

In [2]:
from ColabTurtlePlus.Turtle import *
import random
import ipywidgets as widgets
from IPython.display import display

clearscreen()
initializeTurtle()
speed(8)

player = Turtle()
player.shape("turtle")
player.color("green")

star = Turtle()
star.hideturtle()
star.penup()

STEP = 30          # how far Py moves per click
TURN = 30          # how many degrees Py turns per click
CATCH_DISTANCE = 25  # how close counts as "caught it!"

score = 0
status = widgets.Output()


def place_new_star():
    star.clear()
    x = random.randint(-300, 300)
    y = random.randint(-200, 200)
    star.goto(x, y)
    star.dot(24, "gold")


def show_status(message=""):
    with status:
        status.clear_output()
        remaining = round(player.distance(star.xcor(), star.ycor()))
        print(f"⭐ Score: {score}    Distance to star: {remaining}")
        if message:
            print(message)


def check_catch():
    global score
    remaining = player.distance(star.xcor(), star.ycor())
    if remaining < CATCH_DISTANCE:
        score += 1
        celebrate(f"You caught the star! Score: {score}")
        place_new_star()
    show_status()


def on_forward(_):
    player.forward(STEP)
    check_catch()


def on_left(_):
    player.left(TURN)
    show_status()


def on_right(_):
    player.right(TURN)
    show_status()


def on_new_game(_):
    global score
    score = 0
    player.clear()
    player.penup()
    player.goto(0, 0)
    player.setheading(90)
    player.pendown()
    place_new_star()
    show_status("New game! Go catch the star!")


forward_button = widgets.Button(description="⬆️ Forward", button_style="success")
left_button = widgets.Button(description="↩️ Turn Left")
right_button = widgets.Button(description="↪️ Turn Right")
new_game_button = widgets.Button(description="🔄 New Game", button_style="warning")

forward_button.on_click(on_forward)
left_button.on_click(on_left)
right_button.on_click(on_right)
new_game_button.on_click(on_new_game)

place_new_star()
show_status("Use the buttons to catch the star!")
display(widgets.HBox([left_button, forward_button, right_button]), new_game_button, status)


Put clearscreen() as the first line in a cell (after the import command) to re-run turtle commands in the cell


Button(button_style='warning', description='🔄 New Game', style=ButtonStyle())

Output(outputs=({'name': 'stdout', 'text': '⭐ Score: 0    Distance to star: 288\nUse the buttons to catch the …

In [3]:
from ColabTurtlePlus.Turtle import *
import random
import ipywidgets as widgets
from IPython.display import display

clearscreen()
initializeTurtle()
speed(8)

player = Turtle()
player.shape("turtle")
player.color("green")

star = Turtle()
star.hideturtle()
star.penup()

STEP = 30
TURN = 30
CATCH_DISTANCE = 25
COLLISION_BUFFER = 6   # how close to an old line counts as "touching" it

score = 0
game_over = False
trail = []  # list of segments: ((x1, y1), (x2, y2))

status = widgets.Output()


def place_new_star():
    star.clear()
    x = random.randint(-300, 300)
    y = random.randint(-200, 200)
    star.goto(x, y)
    star.dot(24, "gold")


def show_status(message=""):
    with status:
        status.clear_output()
        remaining = round(player.distance(star.xcor(), star.ycor()))
        print(f"⭐ Score: {score}    Distance to star: {remaining}")
        if message:
            print(message)


def check_catch():
    global score
    remaining = player.distance(star.xcor(), star.ycor())
    if remaining < CATCH_DISTANCE:
        score += 1
        show_status(f"You caught the star! Score: {score}")
        place_new_star()
    else:
        show_status()


# ---------- collision helpers ----------

def ccw(a, b, c):
    return (c[1] - a[1]) * (b[0] - a[0]) > (b[1] - a[1]) * (c[0] - a[0])


def segments_intersect(p1, p2, p3, p4):
    """True if segment p1-p2 crosses segment p3-p4."""
    return (ccw(p1, p3, p4) != ccw(p2, p3, p4)) and \
           (ccw(p1, p2, p3) != ccw(p1, p2, p4))


def point_segment_distance(p, a, b):
    """Shortest distance from point p to segment a-b."""
    px, py = p
    ax, ay = a
    bx, by = b
    dx, dy = bx - ax, by - ay
    if dx == 0 and dy == 0:
        return ((px - ax) ** 2 + (py - ay) ** 2) ** 0.5
    t = ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)
    t = max(0, min(1, t))
    cx, cy = ax + t * dx, ay + t * dy
    return ((px - cx) ** 2 + (py - cy) ** 2) ** 0.5


def hits_trail(new_start, new_end):
    """Check the new move against every old segment except the one
    right before it (which always shares an endpoint, so it would
    otherwise trigger a false positive)."""
    for seg in trail[:-1]:
        if segments_intersect(new_start, new_end, seg[0], seg[1]):
            return True
        if point_segment_distance(new_end, seg[0], seg[1]) < COLLISION_BUFFER:
            return True
    return False


def end_game(message):
    global game_over
    game_over = True
    forward_button.disabled = True
    left_button.disabled = True
    right_button.disabled = True
    show_status(message)


# ---------- button handlers ----------

def on_forward(_):
    if game_over:
        return
    start_pos = (player.xcor(), player.ycor())
    player.forward(STEP)
    end_pos = (player.xcor(), player.ycor())

    if hits_trail(start_pos, end_pos):
        end_game(f"💥 Game Over! You crossed your own trail. Final Score: {score}")
        return

    trail.append((start_pos, end_pos))
    check_catch()


def on_left(_):
    if game_over:
        return
    player.left(TURN)
    show_status()


def on_right(_):
    if game_over:
        return
    player.right(TURN)
    show_status()


def on_new_game(_):
    global score, game_over, trail
    score = 0
    game_over = False
    trail = []
    player.clear()
    player.penup()
    player.goto(0, 0)
    player.setheading(90)
    player.pendown()
    forward_button.disabled = False
    left_button.disabled = False
    right_button.disabled = False
    place_new_star()
    show_status("New game! Go catch the star!")


forward_button = widgets.Button(description="⬆️ Forward", button_style="success")
left_button = widgets.Button(description="↩️ Turn Left")
right_button = widgets.Button(description="↪️ Turn Right")
new_game_button = widgets.Button(description="🔄 New Game", button_style="warning")

forward_button.on_click(on_forward)
left_button.on_click(on_left)
right_button.on_click(on_right)
new_game_button.on_click(on_new_game)

place_new_star()
show_status("Use the buttons to catch the star!")
display(widgets.HBox([left_button, forward_button, right_button]), new_game_button, status)

Button(button_style='warning', description='🔄 New Game', style=ButtonStyle())

Output(outputs=({'name': 'stdout', 'text': '⭐ Score: 0    Distance to star: 161\nUse the buttons to catch the …

In [4]:
from ColabTurtlePlus.Turtle import *
import random
import ipywidgets as widgets
from IPython.display import display

# --- Initialize Screen and Turtles ---
clearscreen()
initializeTurtle()
speed(8)

player = Turtle()
player.shape("turtle")
player.color("green")

star = Turtle()
star.hideturtle()
star.penup()

# --- Game Parameters ---
STEP = 30           # distance moved per click
TURN = 30           # degrees turned per click
CATCH_DISTANCE = 25 # distance threshold to catch star

# --- Game State Variables ---
score = 0
game_over = False
path_segments = []  # Stores list of line segments ((x1, y1), (x2, y2))
status = widgets.Output()


# --- Geometry Helper Functions for Collision Detection ---
def cross_product(A, B, C):
    """Calculates cross product of vector AB and AC."""
    return (B[0] - A[0]) * (C[1] - A[1]) - (B[1] - A[1]) * (C[0] - A[0])


def is_on_segment(P, A, B):
    """Checks if point P lies on line segment AB."""
    return (
        min(A[0], B[0]) <= P[0] <= max(A[0], B[0])
        and min(A[1], B[1]) <= P[1] <= max(A[1], B[1])
        and abs(cross_product(A, B, P)) < 1e-5
    )


def segments_intersect(A, B, C, D):
    """
    Checks if new segment AB intersects existing segment CD.
    Handles general crossings as well as stepping back on past paths.
    """
    cp1 = cross_product(A, B, C)
    cp2 = cross_product(A, B, D)
    cp3 = cross_product(C, D, A)
    cp4 = cross_product(C, D, B)

    # General line crossing check
    if ((cp1 > 0 and cp2 < 0) or (cp1 < 0 and cp2 > 0)) and (
        (cp3 > 0 and cp4 < 0) or (cp3 < 0 and cp4 > 0)
    ):
        return True

    # Check if end point B lands on an old segment
    if is_on_segment(B, C, D):
        return True

    # Check if old segment endpoints lie on new segment AB (excluding starting point A)
    if is_on_segment(C, A, B) and C != A:
        return True
    if is_on_segment(D, A, B) and D != A:
        return True

    return False


# --- Game Logic Functions ---
def place_new_star():
    star.clear()
    x = random.randint(-300, 300)
    y = random.randint(-200, 200)
    star.goto(x, y)
    star.dot(24, "gold")


def show_status(message=""):
    with status:
        status.clear_output()
        remaining = round(player.distance(star.xcor(), star.ycor()))
        print(f"⭐ Score: {score}    Distance to star: {remaining}")
        if message:
            print(message)


def check_catch():
    global score
    remaining = player.distance(star.xcor(), star.ycor())
    if remaining < CATCH_DISTANCE:
        score += 1
        place_new_star()
        show_status(f"🎉 You caught the star! Score: {score}")
    else:
        show_status()


def on_forward(_):
    global game_over

    if game_over:
        show_status("💥 GAME OVER! Click 'New Game' to restart.")
        return

    # Start point of this move
    start_pos = (round(player.xcor(), 2), round(player.ycor(), 2))

    player.forward(STEP)

    # End point of this move
    end_pos = (round(player.xcor(), 2), round(player.ycor(), 2))

    # Check if new segment crosses any previous segment
    for seg in path_segments:
        if segments_intersect(start_pos, end_pos, seg[0], seg[1]):
            game_over = True
            player.color("red")
            show_status("💥 GAME OVER! You crossed your own path!")
            return

    # Save the new valid path segment
    path_segments.append((start_pos, end_pos))

    check_catch()


def on_left(_):
    if game_over:
        show_status("💥 GAME OVER! Click 'New Game' to restart.")
        return
    player.left(TURN)
    show_status()


def on_right(_):
    if game_over:
        show_status("💥 GAME OVER! Click 'New Game' to restart.")
        return
    player.right(TURN)
    show_status()


def on_new_game(_):
    global score, game_over, path_segments
    score = 0
    game_over = False
    path_segments = []

    player.clear()
    player.color("green")
    player.penup()
    player.goto(0, 0)
    player.setheading(90)
    player.pendown()

    place_new_star()
    show_status("New game! Go catch the star without crossing your path!")


# --- UI Controls Setup ---
forward_button = widgets.Button(
    description="⬆️ Forward", button_style="success"
)
left_button = widgets.Button(description="↩️ Turn Left")
right_button = widgets.Button(description="↪️ Turn Right")
new_game_button = widgets.Button(
    description="🔄 New Game", button_style="warning"
)

forward_button.on_click(on_forward)
left_button.on_click(on_left)
right_button.on_click(on_right)
new_game_button.on_click(on_new_game)

# Initialize start position
on_new_game(None)

# Display controls
display(
    widgets.HBox([left_button, forward_button, right_button]),
    new_game_button,
    status,
)

Button(button_style='warning', description='🔄 New Game', style=ButtonStyle())

Output(outputs=({'name': 'stdout', 'text': '⭐ Score: 0    Distance to star: 129\nNew game! Go catch the star w…

## Try it!

Click **⬆️ Forward** and **↩️ / ↪️ Turn** a few times. Watch the distance
number get smaller as you get closer — that's how you know you're going
the right way!

## Change the rules

Games are just code, which means **you** can change how they work. Go
back up to the game cell and try changing:

* `STEP = 30` to `STEP = 60` — bigger steps, faster (but harder to aim!)
* `CATCH_DISTANCE = 25` to `CATCH_DISTANCE = 60` — easier to catch the star
* the numbers in `random.randint(-300, 300)` — make the star appear in a
  smaller area

Then run the cell again to play your new version.

## Quick quiz

In [ ]:
quiz(
    question="In the game, what does `if remaining < CATCH_DISTANCE:` check?",
    choices=[
        "If Py is facing north",
        "If Py is close enough to the star to catch it",
        "If the score is a positive number",
    ],
    correct_index=1,
    hint="remaining holds the distance between Py and the star.",
)


You built and played your own game with Python. Amazing work!

In [ ]:
badge("Game Maker", emoji="⭐")